In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np


In [ ]:
db_path = Path(r"E:\ProyectoAnalisisElectrico\DiaPromedio\Mensuales")
mercado = pd.read_csv(Path(r"E:\ProyectoAnalisisElectrico\PotencialesClientes\CalorVentasRegionales.csv"))

In [ ]:
month_mapping = {
    "enero": "01", "febrero": "02", "marzo": "03", "abril": "04",
    "mayo": "05", "junio": "06", "julio": "07", "agosto": "08",
    "septiembre": "09", "octubre": "10", "noviembre": "11", "diciembre": "12"
}

def get_folders_by_period(start_date=None, end_date=None, meses_filtro=None):
    available_folders = sorted([d.name for d in db_path.iterdir()])
    
    if start_date and end_date:
        start_str = f"{start_date[1] % 100:02d}{month_mapping[start_date[0]]}"
        end_str = f"{end_date[1] % 100:02d}{month_mapping[end_date[0]]}"
        idx_start = available_folders.index(start_str)
        idx_end = available_folders.index(end_str)
        available_folders = available_folders[idx_start : idx_end + 1]
        
    if meses_filtro:
        codigos_permitidos = [month_mapping[m.lower()] for m in meses_filtro]
        available_folders = [f for f in available_folders if f[-2:] in codigos_permitidos]
        
    return available_folders

In [ ]:
estaciones_meses = {
    "Verano": ["enero", "febrero", "marzo"],
    "Otono": ["abril", "mayo", "junio"],
    "Invierno": ["julio", "agosto", "septiembre"],
    "Primavera": ["octubre", "noviembre", "diciembre"]
}
periods = get_folders_by_period(
    start_date=("mayo", 2025), 
    end_date=("abril", 2026), 
    meses_filtro=None
)

print(f"Carpetas seleccionadas: {periods}")

dfs = []
for month in periods:
    df_path = db_path / month / f"{month}_mean_month.parquet"
    dfs.append(pd.read_parquet(df_path))
    
retiros = pd.concat(dfs, ignore_index=True)


In [ ]:
retiros.head()

In [ ]:
retiros['PERIODO'] = pd.to_datetime(retiros['Año_Mes']).dt.strftime('%y%m')
print(retiros[['Año_Mes', 'PERIODO']].head())

In [ ]:
retiros['clave'] = retiros['clave'].astype(str).str.strip()
mercado['clave'] = mercado['clave'].astype(str).str.strip()

retiros['PERIODO'] = retiros['PERIODO'].astype(str).str.strip()
mercado['PERIODO'] = mercado['PERIODO'].astype(str).str.strip()

retiros_con_cliente = pd.merge(
    retiros,
    mercado,
    on=['clave', 'PERIODO'],
    how='inner'
)

retiros_con_cliente = retiros_con_cliente.rename(columns={
    'RUT': 'RUT_PROVEEDOR',            # El que venía en retiros (ej. AES Andes)
    'Razon_Social': 'PROVEEDOR',       # El que venía en retiros
    'RUT_RAZON_SOCIAL': 'RUT_CLIENTE', # El que trajimos de mercado (tu prospecto)
    'REGION': 'REGION_CLIENTE'         # La región oficial del cliente
})

# 4. Revisamos cuántos datos sobrevivieron a la "aniquilación"
print(f"Filas originales en retiros físicos: {len(retiros):,}")
print(f"Filas tras cruzar con clientes válidos: {len(retiros_con_cliente):,}")
print("\nMuestra de los datos cruzados:")
print(retiros_con_cliente[['clave', 'PERIODO', 'Hora', 'medida_mean', 'RUT_PROVEEDOR', 'RUT_CLIENTE']].head())

In [ ]:
retiros_con_cliente.head()

In [ ]:
retiros_con_cliente.columns

In [10]:


metrics = ['medida', 'CMg[CLP/KWh]', 'valorizado_CLP']
for m in metrics:
    retiros_con_cliente[f'{m}_sum_val'] = retiros_con_cliente[f'{m}_mean'] * retiros_con_cliente[f'{m}_count']
    
    std_squared = retiros_con_cliente[f'{m}_std'].fillna(0) ** 2
    retiros_con_cliente[f'{m}_sum_sq'] = (retiros_con_cliente[f'{m}_count'] - 1) * std_squared + retiros_con_cliente[f'{m}_count'] * (retiros_con_cliente[f'{m}_mean'] ** 2)

def crear_log(x):
    return "::".join(x.dropna().astype(str).unique())


group_cols = ['clave', 'RUT_CLIENTE', 'REGION_CLIENTE', 'macrozona', 'Zona', 'Hora']

agg_rules = {
    # --- MÉTRICAS ELÉCTRICAS A RECONSTRUIR ---
    'medida_sum_val': ('medida_sum_val', 'sum'),
    'medida_sum_sq': ('medida_sum_sq', 'sum'),
    'medida_count': ('medida_count', 'sum'),
    
    'CMg_sum_val': ('CMg[CLP/KWh]_sum_val', 'sum'),
    'CMg_sum_sq': ('CMg[CLP/KWh]_sum_sq', 'sum'),
    'CMg_count': ('CMg[CLP/KWh]_count', 'sum'),
    
    'valorizado_sum_val': ('valorizado_CLP_sum_val', 'sum'),
    'valorizado_sum_sq': ('valorizado_CLP_sum_sq', 'sum'),
    'valorizado_count': ('valorizado_CLP_count', 'sum'),

    # --- IDENTIDAD DEL PROSPECTO TÉRMICO (Variables Estáticas) ---
    'CLIENTE': ('CLIENTE', 'last'),
    'CLIENTE_log': ('CLIENTE', crear_log),
    'n_clientes': ('CLIENTE', 'nunique'),
    
    'TIPO': ('TIPO', 'last'),  # <-- Simplificado, sin logs ni conteos
    
    'NOMBRE_ESTABLECIMIENTO': ('NOMBRE_ESTABLECIMIENTO', 'last'), # <-- Simplificado
    "SECTOR": ('SECTOR', crear_log), 
    "SUBSECTOR": ('SUBSECTOR', crear_log), 
    "RUBRO": ('RUBRO', "last"), # <-- Simplificado

    
    # --- DEMANDA TÉRMICA (Métricas fijas) ---
    'DEMANDA_CALOR_MWH_sum': ('DEMANDA_CALOR_MWH_sum', 'last'),
    'DEMANDA_CALOR_MWH_mean': ('DEMANDA_CALOR_MWH_mean', 'last'),
    'DEMANDA_CALOR_MWH_std': ('DEMANDA_CALOR_MWH_std', 'last'),
    'DEMANDA_CALOR_MWH_max': ('DEMANDA_CALOR_MWH_max', 'last'),
    'DEMANDA_CALOR_MWH_min': ('DEMANDA_CALOR_MWH_min', 'last'),
    
    # --- TRAZABILIDAD DEL PROVEEDOR ELÉCTRICO ---
    'RUT_PROVEEDOR': ('RUT_PROVEEDOR', 'last'),
    'RUT_PROVEEDOR_log': ('RUT_PROVEEDOR', crear_log),
    'n_rut_proveedores': ('RUT_PROVEEDOR', 'nunique'),

    'PROVEEDOR': ('PROVEEDOR', 'last'),
    'PROVEEDOR_log': ('PROVEEDOR', crear_log),
    'n_proveedores': ('PROVEEDOR', 'nunique'),
    
    # --- DATOS FÍSICOS DE LA BARRA ---
    'nombre_barra': ('nombre_barra', 'last'),
    'nombre_barra_log': ('nombre_barra', crear_log),
    'n_nombres_barra': ('nombre_barra', 'nunique'),
    
    'tension': ('tension', 'last'),
    'tension_log': ('tension', crear_log),
    'n_tensiones': ('tension', 'nunique'),
    
    'Nombre_Corto': ('Nombre_Corto', 'last'),
    'Nombre_Corto_log': ('Nombre_Corto', crear_log),
    'n_nombres_cortos': ('Nombre_Corto', 'nunique'),
    
    # --- CONTROL TEMPORAL ---
    'periodo_last': ('PERIODO', 'last'),
    'periodos_log': ('PERIODO', crear_log),      
    'meses_operados': ('PERIODO', 'nunique')     
}

# Ejecutamos el GroupBy
perfil_promedio_final = retiros_con_cliente.groupby(group_cols).agg(**agg_rules).reset_index()

## 4. Cálculo final de Promedios y Desviaciones Estándar Combinadas (CORREGIDO)
prefix_map = ['medida', 'CMg', 'valorizado']

for m, prefix in zip(metrics, prefix_map):
    # Promedio Combinado
    perfil_promedio_final[f'{m}_mean'] = perfil_promedio_final[f'{prefix}_sum_val'] / perfil_promedio_final[f'{prefix}_count']
    
    # Varianza Combinada y Desviación Estándar
    variance = (perfil_promedio_final[f'{prefix}_sum_sq'] - (perfil_promedio_final[f'{prefix}_sum_val'] ** 2 / perfil_promedio_final[f'{prefix}_count'])) / (perfil_promedio_final[f'{prefix}_count'] - 1)
    perfil_promedio_final[f'{m}_std'] = np.sqrt(np.maximum(0, variance))
    
    # Restauramos el conteo con el nombre correcto SOLO si son distintos
    if f'{m}_count' != f'{prefix}_count':
        perfil_promedio_final[f'{m}_count'] = perfil_promedio_final[f'{prefix}_count']
        perfil_promedio_final = perfil_promedio_final.drop(columns=[f'{prefix}_count'])
    
    # Limpieza: SOLO borramos las sumas temporales, NUNCA el count de 'medida'
    perfil_promedio_final = perfil_promedio_final.drop(columns=[f'{prefix}_sum_val', f'{prefix}_sum_sq'])

# 5. Total de energía
perfil_promedio_final['medida_total'] = perfil_promedio_final.groupby(['clave', 'RUT_CLIENTE', 'REGION_CLIENTE'])['medida_mean'].transform('sum')

In [11]:
perfil_promedio_final.to_parquet(Path(r"E:\ProyectoAnalisisElectrico\PotencialesClientes\PerfilesMercado.parquet"), index=False)

In [12]:
perfil_promedio_final.columns

Index(['clave', 'RUT_CLIENTE', 'REGION_CLIENTE', 'macrozona', 'Zona', 'Hora',
       'medida_count', 'CLIENTE', 'CLIENTE_log', 'n_clientes', 'TIPO',
       'NOMBRE_ESTABLECIMIENTO', 'SECTOR', 'SUBSECTOR', 'RUBRO',
       'DEMANDA_CALOR_MWH_sum', 'DEMANDA_CALOR_MWH_mean',
       'DEMANDA_CALOR_MWH_std', 'DEMANDA_CALOR_MWH_max',
       'DEMANDA_CALOR_MWH_min', 'RUT_PROVEEDOR', 'RUT_PROVEEDOR_log',
       'n_rut_proveedores', 'PROVEEDOR', 'PROVEEDOR_log', 'n_proveedores',
       'nombre_barra', 'nombre_barra_log', 'n_nombres_barra', 'tension',
       'tension_log', 'n_tensiones', 'Nombre_Corto', 'Nombre_Corto_log',
       'n_nombres_cortos', 'periodo_last', 'periodos_log', 'meses_operados',
       'medida_mean', 'medida_std', 'CMg[CLP/KWh]_mean', 'CMg[CLP/KWh]_std',
       'CMg[CLP/KWh]_count', 'valorizado_CLP_mean', 'valorizado_CLP_std',
       'valorizado_CLP_count', 'medida_total'],
      dtype='str')

In [13]:
perfil_promedio_final[perfil_promedio_final["meses_operados"]==3]

,clave,RUT_CLIENTE,REGION_CLIENTE,macrozona,Zona,Hora,medida_count,CLIENTE,CLIENTE_log,n_clientes,...,meses_operados,medida_mean,medida_std,CMg[CLP/KWh]_mean,CMg[CLP/KWh]_std,CMg[CLP/KWh]_count,valorizado_CLP_mean,valorizado_CLP_std,valorizado_CLP_count,medida_total
456,1088377_PEAJE_PBC,96545900-6,Metropolitana de Santiago,Centro,Norte Distribución,0,90,PANIMEX QUIMICA LIMITADA,PANIMEX QUIMICA LIMITADA,1,...,3,-691.088889,153.404809,55.140900,18.541867,90,-37894.338709,15174.078223,90,-16484.291111
457,1088377_PEAJE_PBC,96545900-6,Metropolitana de Santiago,Centro,Norte Distribución,1,90,PANIMEX QUIMICA LIMITADA,PANIMEX QUIMICA LIMITADA,1,...,3,-690.711111,153.875473,56.480256,22.710573,90,-38609.851542,16903.974650,90,-16484.291111
458,1088377_PEAJE_PBC,96545900-6,Metropolitana de Santiago,Centro,Norte Distribución,2,90,PANIMEX QUIMICA LIMITADA,PANIMEX QUIMICA LIMITADA,1,...,3,-690.261111,154.439497,56.369776,19.230553,90,-38281.308732,13619.167068,90,-16484.291111
459,1088377_PEAJE_PBC,96545900-6,Metropolitana de Santiago,Centro,Norte Distribución,3,90,PANIMEX QUIMICA LIMITADA,PANIMEX QUIMICA LIMITADA,1,...,3,-688.442222,157.003839,56.575945,20.997332,90,-39040.690170,17927.760829,90,-16484.291111
460,1088377_PEAJE_PBC,96545900-6,Metropolitana de Santiago,Centro,Norte Distribución,4,90,PANIMEX QUIMICA LIMITADA,PANIMEX QUIMICA LIMITADA,1,...,3,-683.906667,157.604012,57.701308,20.000611,90,-39467.119513,15947.830837,90,-16484.291111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18259,STANDREWSDG_TRASDAL,96783150-6,Los Lagos,Sur,Sur Distribución,19,91,ST ANDREWS SMOKY DELICACIES S.A.,ST ANDREWS SMOKY DELICACIES S.A.,1,...,3,0.000000,0.000000,78.895785,53.448578,91,0.000000,0.000000,91,-44.440220
18260,STANDREWSDG_TRASDAL,96783150-6,Los Lagos,Sur,Sur Distribución,20,91,ST ANDREWS SMOKY DELICACIES S.A.,ST ANDREWS SMOKY DELICACIES S.A.,1,...,3,0.000000,0.000000,75.891155,52.947522,91,0.000000,0.000000,91,-44.440220
18261,STANDREWSDG_TRASDAL,96783150-6,Los Lagos,Sur,Sur Distribución,21,91,ST ANDREWS SMOKY DELICACIES S.A.,ST ANDREWS SMOKY DELICACIES S.A.,1,...,3,0.000000,0.000000,74.518421,54.034346,91,0.000000,0.000000,91,-44.440220
18262,STANDREWSDG_TRASDAL,96783150-6,Los Lagos,Sur,Sur Distribución,22,91,ST ANDREWS SMOKY DELICACIES S.A.,ST ANDREWS SMOKY DELICACIES S.A.,1,...,3,0.000000,0.000000,80.172907,57.510912,91,0.000000,0.000000,91,-44.440220


In [14]:
print("--- AUDITORÍA DE INTEGRIDAD EXACTA ---")

# 1. Filtramos solo la Hora 0 para auditar a cada prospecto una sola vez
auditoria = perfil_promedio_final[perfil_promedio_final['Hora'] == 0].copy()

# 2. Función que lee "2601::2602", identifica los meses y suma sus días calendario exactos
def sumar_dias_calendario(periodos_log):
    periodos = str(periodos_log).split("::")
    # pd.to_datetime('%y%m') convierte '2601' a enero 2026 y saca que tiene 31 días
    return sum(pd.to_datetime(p, format='%y%m').days_in_month for p in periodos)

# 3. Calculamos la suma exacta de días que DEBERÍA tener según sus meses de operación
auditoria['dias_teoricos'] = auditoria['periodos_log'].apply(sumar_dias_calendario)

# 4. Comparamos la realidad física vs la teoría del calendario (BUSCANDO CUALQUIER DIFERENCIA)
anomalias = auditoria[auditoria['medida_count'] != auditoria['dias_teoricos']]

if anomalias.empty:
    print("TEST PASADO: El conteo de mediciones cuadra perfecto con los días del calendario. No hay registros inflados ni datos faltantes.")
else:
    print(f"ALERTA: Se detectaron {len(anomalias)} perfiles con discrepancias entre mediciones y días calendario.")
    print("\nDetalle de la anomalía (revisa si medida_count es mayor o menor al teórico):")
    print(anomalias[['clave', 'RUT_CLIENTE', 'periodos_log', 'medida_count', 'dias_teoricos']].head(10))

--- AUDITORÍA DE INTEGRIDAD EXACTA ---
TEST PASADO: El conteo de mediciones cuadra perfecto con los días del calendario. No hay registros inflados ni datos faltantes.
